<a href="https://colab.research.google.com/github/ThayLoser/ollama-streamlit-colab/blob/main/StreamlitOllamaVietnamese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. **Cài đặt Ollama Dependencies**

In [20]:
%%capture
!sudo apt-get install zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

2. **Khởi động Ollama**


In [21]:
%%capture
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

3. **Tải model Gemma3:4B**

In [22]:
%%capture
!ollama pull gemma3:4b

4. **Cài đặt thư viện**

In [23]:
%%capture
!pip install streamlit
!pip install ollama

5. **Mã nguồn**

In [24]:
%%writefile app.py

import streamlit as st
import time
import uuid
import ollama

st.title("Ứng dụng Web Chatbot", text_alignment="center")
st.markdown("*Ứng dụng chatbot local bằng :rainbow[Streamlit], sử dụng :rainbow[Ollama] với mô hình :rainbow[gemma3:4B]*", text_alignment="center")

if "model" not in st.session_state:
  st.session_state["model"] = "gemma3:4b"

if "chat_sessions" not in st.session_state:
  st.session_state.chat_sessions = {}

if "current_chat_id" not in st.session_state:
  new_session_id = str(uuid.uuid4())
  st.session_state.current_chat_id = new_session_id
  st.session_state.chat_sessions[new_session_id] = []

with st.sidebar:
  with st.expander(label="**Thông tin sinh viên**", icon=":material/info:"):
      st.markdown("""
          **Họ và tên**: Nguyễn Anh Thái

          **Mã số sinh viên**: 24120224

          **Học phần**: Tư duy tính toán

          **Nhóm**: CQ2024/2 [N1]
      """)

  with st.expander(label="**Thông tin model**", icon=":material/android:"):
    st.markdown("[**Gemma**](https://ollama.com/library/gemma3) is a lightweight, family of models from Google built on Gemini technology. The Gemma 3 models are multimodal—processing text and images—and feature a 128K context window with support for over 140 languages. Available in 270M, 1B, 4B, 12B, and 27B parameter sizes, they excel in tasks like question answering, summarization, and reasoning, while their compact design allows deployment on resource-limited devices.")

  if st.button("Cuộc trò chuyện mới", width="stretch"):
    new_session_id = str(uuid.uuid4())
    st.session_state.current_chat_id = new_session_id
    st.session_state.chat_sessions[new_session_id] = []
    st.rerun()

  st.divider()

  for session_id, chat_msgs in reversed(st.session_state.chat_sessions.items()):
    if not chat_msgs and session_id != st.session_state.current_chat_id:
      continue

    if chat_msgs:
      first_user_msg = next((msg["content"] for msg in chat_msgs if msg["role"] == "user" and "content" in msg), "Chat")
      title = first_user_msg[:25] + "..." if len(first_user_msg) > 25 else first_user_msg
    else:
      title = "Cuộc trò chuyện"

    is_active = session_id == st.session_state.current_chat_id
    btn_label = f"{title}" if is_active else title

    if st.button(btn_label, key=session_id, width="stretch", disabled=is_active):
      st.session_state.current_chat_id = session_id
      st.rerun()

messages = st.session_state.chat_sessions[st.session_state.current_chat_id]

for message in messages:
  with st.chat_message(message["role"]):
    if message.get("files"):
      st.image(message["files"])
    if message.get("content"):
      st.markdown(message["content"])

prompt = st.chat_input(placeholder="Hỏi Gemma3:4B", accept_file=True, file_type=["jpg", "jpeg", "png"])

if prompt:
  text_input = prompt.text
  image_input = prompt.files

  user_msg = {"role": "user", "content": text_input}
  if image_input:
    user_msg["files"] = image_input[0]

  st.session_state.chat_sessions[st.session_state.current_chat_id].append(user_msg)

  with st.chat_message("user"):
    if image_input:
      st.image(image_input[0])
    if text_input:
      st.markdown(text_input)

  with st.chat_message("assistant"):
    ollama_messages = [
      {"role": "system", "content": "Bạn là một giảng viên lớp Tư duy tính toán của trường đại học khoa học tự nhiên, đại học quốc gia thành phố hồ chí minh, việt nam"}
    ]

    for message in st.session_state.chat_sessions[st.session_state.current_chat_id]:
      clean = {"role": message["role"], "content": message["content"]}

      if message.get("files"):
        try:
          clean["images"] = [message["files"].getvalue()]
        except AttributeError:
          pass

      ollama_messages.append(clean)

    whole_response = ollama.chat(model=st.session_state["model"], messages=ollama_messages, stream=True)

    def stream_text():
      for chunk in whole_response:
        time.sleep(0.05)
        yield chunk["message"]["content"]

    full_response = st.write_stream(stream_text)
    st.session_state.chat_sessions[st.session_state.current_chat_id].append({"role": "assistant", "content": full_response})

Overwriting app.py


6. **Sử dụng Cloudflare để lấy link ứng dụng**

In [25]:
!wget -q -O cloudflared-linux-amd64 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

!streamlit run app.py &>/dev/null&

import time
import subprocess
import re

print("Khởi động Cloudflare Tunnel")

process = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8501'],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    if "trycloudflare.com" in line:
        url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if url:
            print(f"Link:\n{url.group(0)}")
            break

cloudflared-linux-amd64: Text file busy
Khởi động Cloudflare Tunnel
Link:
https://brought-entrance-layers-cingular.trycloudflare.com
